### Population growth, environmental resource strain, housing developments

- <b>Environmental Research Question / Intro & Background:</b>
	- Sustainable economic development = balancing environmental pressures + economic needs
    - Housing affordability is a big news item these days
    - Two options: 
    	- Denser urban housing that puts a larger strain on affected area but leaves more area untouched vs
        - Sparser rural housing that puts a lower strain on affected area but develops more area
    - Cite recent news items where those two were in tension
    - Cite flagship policies where governments are explicitly trying to manage the two
- <b>Data:</b>
	- Clean 2000-2020 Census datasets (x3)
    	- **Filter** to county-level rows only
        - **Select** the columns I want
        - **Rename** the columns to something descriptive
        - **Pivot longer** the dataset to a format I am more familiar in working with
        - I can be as granular as I want here - the point is I'm making a to-do list for myself
    - Clean 1990-2000 Census dataset
    	- Download all the different .txt files
        - Write code to read in only the rows that I want
        - Write code to make columns consistent with 2000-2020
     - Look up how the US Census keeps population data series consistent across decades
     - Read in housing prices dataset
     	- This seems relatively easier to deal with 
- <b>Exploratory Data Analysis:</b>
	- Have some text explaining how a new audience should be taking in all this information
    - Line graph of US population by time
    - Line graph of each state's population by time
    - Line graph of housing prices by time
    - Histogram of population growth percentage in each year (to see what the typical population growth rate is, and how it's distributed)
- <b>Final Analysis and Visualizations:</b>
	- Heat map of areas experiencing high population growth / loss
    - Heat map of areas experiencing high housing inflation
- <b>Ethical Considerations / Limitations:</b>
- <b>Big picture takeaways / Future directions:</b>
-  <b>Notes for reproducibility:</b>
	- We'll leave these last 3 sections to the very end of my work

In [3]:
# copied over my initial read data line from my "capstone beginning file"
library(tidyverse)
cs20 <- read.csv('data/co-est2020-alldata.csv')
head(cs20)
#start pseudocoding based on my todo list
cs20_cleaned <- cs20 %>%
filter() %>%
select() %>%
pivot_longer()
cs20_cleaned

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,⋯,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015,RNETMIG2016,RNETMIG2017,RNETMIG2018,RNETMIG2019,RNETMIG2020
,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<int>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,40,3,6,1,0,Alabama,Alabama,4779736,4780118,4785514,⋯,0.6800098,1.282825,1.6143914,0.6538283,0.7304192,0.8543837,1.197150,1.9660955,2.6778663,3.091308
2,50,3,6,1,1,Alabama,Autauga County,54571,54582,54761,⋯,6.2369306,-5.971016,-3.7733441,2.2066400,-1.5297064,4.9544032,0.993228,-0.0180211,3.4860110,6.290545
3,50,3,6,1,3,Alabama,Baldwin County,182265,182263,183121,⋯,16.7054368,17.670696,22.9242875,20.3000883,17.9022731,21.4364985,22.476720,24.8463353,25.2425074,26.401562
4,50,3,6,1,5,Alabama,Barbour County,27457,27454,27325,⋯,0.3292542,-6.860371,-8.0934255,-5.0638567,-15.6779980,-18.3778392,-25.138734,-8.7901550,-6.2570644,0.649799
5,50,3,6,1,7,Alabama,Bibb County,22915,22904,22858,⋯,-4.9129271,-3.789130,-5.8006952,1.4206122,1.2862022,-0.8417695,-3.235672,-7.2715917,0.2689799,-7.199262
6,50,3,6,1,9,Alabama,Blount County,57322,57322,57372,⋯,0.3480289,-1.597971,-0.2777416,-1.9971172,-1.3035430,-1.2171585,6.193186,0.2422753,0.9341752,1.192544


In [0]:
# helper func for reading census data
read_census <- function(filename){
  df <- read.csv(filename) %>%
    # keep county-level only
    filter(SUMLEV!=40) %>%
    # generate fips code
    mutate(fips=str_pad(STATE*1000+COUNTY,5,pad='0'))%>%
    # keep only state name, county name, fips code, pop vars
    select(-(1:5)) %>% 
    rename(state=STNAME,county=CTYNAME) %>%
    mutate(state=str_squish(state),
           county=str_squish(county))
  df <- df %>%
    # coerce any character cols to numeric cols 
    # this happens for the census estimates every decade
    mutate(across(3:ncol(df), as.numeric)) %>%
    # pivot to county-var-year level
    pivot_longer(!c(state,county,fips)) %>%
    # split col names into var and year
    mutate(var=tolower(str_replace_all(name, "\\d","")),
           year=as.numeric(tolower(str_replace_all(name, "\\D","")))) %>%
    select(-name)
  return(df)
}